<a href="https://colab.research.google.com/github/CevdetSatarr/FlyRank-intern/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CevdetSatarr/FlyRank-intern/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

**Lane 2, full-depth companion to ML-05.** Where ML-05 checked two signals quickly to build a baseline, this notebook does the fuller version the `auditing-signals` skill describes: distributions first, heavy-tail handling before correlating, three safe signal tests, one flag-linked test, all with visible n's.

> Read `skills/README.md`, then load `auditing-signals` + `flyrank/flyrank-data` before working this notebook.

## 0. Connect

Same month (`month=2026-03`), same base frame as ML-05/ML-07 — this audit runs on the exact data the model and baseline already use.

In [1]:
%pip -q install duckdb
import os, duckdb, pandas as pd, numpy as np

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

base = con.sql("""
    SELECT
        f.client_hash_id, f.content_hash_id,
        SUM(f.gsc_impressions) AS gsc_impressions,
        AVG(f.gsc_avg_position) AS gsc_avg_position,
        ANY_VALUE(DATE_DIFF('day', c.content_created_date, DATE '2026-03-31')) AS content_age_days,
        ANY_VALUE(c.search_volume) AS search_volume,
        SUM(CASE WHEN EXTRACT(DAY FROM f.report_date) <= 15 THEN f.gsc_clicks ELSE 0 END) AS first_half_clicks,
        SUM(CASE WHEN EXTRACT(DAY FROM f.report_date) > 15  THEN f.gsc_clicks ELSE 0 END) AS second_half_clicks
    FROM {FACT} f LEFT JOIN {DIM_CONTENT} c ON c.content_hash_id = f.content_hash_id
    GROUP BY 1, 2 HAVING SUM(f.gsc_impressions) >= 100
""".format(FACT=FACT, DIM_CONTENT=DIM_CONTENT)).df()

base = base.dropna(subset=['content_age_days', 'search_volume'])
base = base[base['first_half_clicks'] > 0].copy()
base['click_trend_pct'] = (base['second_half_clicks'] - base['first_half_clicks']) / base['first_half_clicks'] * 100
base['is_declining'] = (base['click_trend_pct'] < 0).astype(int)
print(f'{len(base):,} rows ready for audit')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

48,715 rows ready for audit


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Checking `gsc_impressions`, `gsc_avg_position`, `content_age_days`, and `search_volume` before any correlation. Traffic-like fields (`gsc_impressions`, `search_volume`) are expected to be heavy-tailed — a few very large pages, a long tail of small ones — which would distort a plain Pearson correlation. Position and age are expected to be closer to normal/uniform.

In [2]:
desc = base[['gsc_impressions', 'gsc_avg_position', 'content_age_days', 'search_volume']].describe(
    percentiles=[.1, .25, .5, .75, .9, .99]
)
print(desc)

# Heavy-tail check: how far is the mean pulled above the median?
for col in ['gsc_impressions', 'search_volume']:
    mean_, median_ = base[col].mean(), base[col].median()
    print(f'{col}: mean={mean_:.1f}, median={median_:.1f}, mean/median ratio={mean_/median_:.2f}')


       gsc_impressions  gsc_avg_position  content_age_days  search_volume
count     48715.000000      48715.000000      48715.000000        48715.0
mean       4925.603798         10.441105        188.301652      91.672585
std        9412.744604          9.530057        119.749232    1040.007689
min         100.000000          0.103688         18.000000            0.0
10%         401.000000          2.800551         47.000000            0.0
25%         912.000000          4.128743         74.000000            0.0
50%        2276.000000          6.782086        187.000000           10.0
75%        5166.500000         13.345474        263.000000           20.0
90%       11205.200000         24.552939        375.000000           90.0
99%       41676.780000         43.145289        467.000000         1300.0
max      617124.000000         83.963812        494.000000        90500.0
gsc_impressions: mean=4925.6, median=2276.0, mean/median ratio=2.16
search_volume: mean=91.7, median=10.0, mean/

**Reading this before moving on:** if the mean/median ratio for `gsc_impressions` or `search_volume` is well above 1 (a common result for traffic data), that confirms a heavy right tail — a handful of very large pages pulling the mean far above the typical page. That's the signal to use `log1p()` or rank-based comparisons below instead of raw Pearson correlation, per the skill's rule.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Each test below prints a bucket table with visible `n`. **Do not assign a verdict from any bucket under n=50** (n=30 for the cross-cut in Test 3) — write ‘insufficient data’ for that bucket instead, per the skill's sample-size floor.

**Test 1 verdict: MIXED** — “Higher-traffic pages are less likely to be declining this month.” Decline rate drops cleanly from the lowest to the third impressions bucket (0.683 → 0.545 → 0.458, n≈12,000 each), which supports the claim — but the highest-impression bucket ticks back up to 0.483 instead of continuing down. Directionally true for most of the range, not a clean monotonic pattern.

**Test 2 verdict: CONFIRMED** — “Pages ranked further down the results are more likely to be declining.” Clean, monotonic increase across every tier: top_3 0.479 → page_1 0.517 → striking 0.599 → page_3_5 0.612 → deep 0.805, with every bucket (including deep, n=200) well above the sample-size floor.

**Test 3 verdict: FALSE** — “`search_volume` (an external keyword-level estimate) tracks this month's real observed impressions.” No usable pattern across buckets (5,241 → 4,084 → 4,888 → 5,416 — not increasing), and the Spearman rank correlation is −0.014, effectively zero. The external volume estimate carries no real relationship to this month's actual demand in this data.

In [4]:
# Test 1: gsc_impressions (log-bucketed, heavy tail) vs is_declining rate
base['log_impressions'] = np.log1p(base['gsc_impressions'])
imp_bins = base['log_impressions'].quantile([0, .25, .5, .75, 1.0]).tolist()
imp_bins[0] -= 0.01
base['impressions_bucket'] = pd.cut(base['log_impressions'], bins=imp_bins, labels=['q1_low', 'q2', 'q3', 'q4_high'])

test1 = base.groupby('impressions_bucket', observed=True).agg(
    n=('is_declining', 'size'), decline_rate=('is_declining', 'mean')
).round(3)
print('Test 1 — impressions (log-bucketed) vs decline rate')
print(test1)


Test 1 — impressions (log-bucketed) vs decline rate
                        n  decline_rate
impressions_bucket                     
q1_low              12185         0.683
q2                  12179         0.545
q3                  12172         0.458
q4_high             12179         0.483


In [5]:
# Test 2: gsc_avg_position (tiered) vs is_declining rate
def position_tier(p):
    if p <= 3: return 'top_3'
    if p <= 10: return 'page_1'
    if p <= 20: return 'striking'
    if p <= 50: return 'page_3_5'
    return 'deep'

base['position_tier'] = base['gsc_avg_position'].apply(position_tier)
test2 = base.groupby('position_tier', observed=True).agg(
    n=('is_declining', 'size'), decline_rate=('is_declining', 'mean')
).round(3).reindex(['top_3', 'page_1', 'striking', 'page_3_5', 'deep'])
print('Test 2 — position tier vs decline rate')
print(test2)


Test 2 — position tier vs decline rate
                   n  decline_rate
position_tier                     
top_3           5907         0.479
page_1         26713         0.517
striking        8908         0.599
page_3_5        6987         0.612
deep             200         0.805


In [6]:
# Test 3: search_volume tier vs REAL observed gsc_impressions this month
vol_bins = base['search_volume'].quantile([0, .25, .5, .75, 1.0]).tolist()
vol_bins[0] -= 1
base['volume_bucket'] = pd.cut(base['search_volume'], bins=vol_bins, labels=['q1_low', 'q2', 'q3', 'q4_high'], duplicates='drop')

test3 = base.groupby('volume_bucket', observed=True).agg(
    n=('gsc_impressions', 'size'), avg_real_impressions=('gsc_impressions', 'mean')
).round(1)
print('Test 3 — search_volume tier vs real observed impressions this month')
print(test3)

# Rank correlation (Spearman), safe for heavy-tailed data per the skill's rule
corr = base[['search_volume', 'gsc_impressions']].corr(method='spearman').iloc[0, 1]
print(f'\nSpearman rank correlation (search_volume vs real impressions): {corr:.3f}')


Test 3 — search_volume tier vs real observed impressions this month
                   n  avg_real_impressions
volume_bucket                             
q1_low         18775                5241.5
q2             13647                4084.3
q3              4633                4888.4
q4_high        11660                5416.4

Spearman rank correlation (search_volume vs real impressions): -0.014


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

**Signal: staleness (`content_age_days`), behind FlyRank's refresh flags.** Assumption: older content is more likely to be the declining, refresh-worthy kind. This is the same flag-linked signal ML-05's baseline used — audited here with the full bucket-table-plus-verdict method rather than a quick check.

**Flag-linked verdict: FALSE.** Decline rate is essentially flat across every age bucket (<90d: 0.546, 90–365d: 0.540, 365–730d: 0.540 — no `730d+` rows survived this month's slice). All buckets have large, solid n (15k–27k). This directly contradicts the staleness assumption behind the refresh flag on this month's data: older content in this slice is not more likely to be declining than younger content. That's a real, uncomfortable finding, not a data error — and it complicates the ‘Declining Veteran’ archetype built in ML-10, which leaned on age as one of its three defining signals.

In [7]:
age_bins = [0, 90, 365, 730, 100000]
age_labels = ['<90d', '90-365d', '365-730d', '730d+']
base['age_bucket'] = pd.cut(base['content_age_days'], bins=age_bins, labels=age_labels)

flag_test = base.groupby('age_bucket', observed=True).agg(
    n=('is_declining', 'size'), decline_rate=('is_declining', 'mean')
).round(3)
print('Flag-linked test — staleness vs decline rate')
print(flag_test)


Flag-linked test — staleness vs decline rate
                n  decline_rate
age_bucket                     
<90d        15329         0.546
90-365d     27440         0.540
365-730d     5946         0.540


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

Position is the one signal that held up cleanly this month — pages further from the top results really are more likely to be declining, and a review queue built around position tier is on solid ground. Content age is not: despite being the assumption behind FlyRank's own refresh flags, older content in this month's data was no more likely to be declining than newer content, so a refresh queue built purely on ‘this is old’ would be prioritizing on a signal that isn't actually there this month. The external `search_volume` estimate should not be trusted as a stand-in for real demand at all — it showed essentially zero relationship to actual observed impressions, so any prioritization currently leaning on it should switch to real, measured impressions instead.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ x] Every section above is filled — markdown thinking AND the code that backs it
- [ x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x] No client names, URLs, or private queries anywhere
- [ x] My claims use careful words: observed, measured, directional, decision-support
- [ x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.